In [1]:
import tensorflow as tf
import numpy as np

2025-04-15 09:27:31.010764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744720051.033420   32186 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744720051.037737   32186 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-15 09:27:31.053197: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [13]:
def generate_time_series(batch_size, n_steps):
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1)
    time = np.linspace(0, 1, n_steps)
    series = 0.5 * np.sin((time - offsets1) * (freq1 * 10 + 10))
    series += 2 * np.sin((time - offsets2) * (freq2 * 20 + 20))
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5)

    return series[..., np.newaxis].astype(np.float32)

In [14]:
n_steps = 50

series = generate_time_series(10_000, n_steps + 1)

X_train, y_train = series[:7000, :n_steps], series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_test, y_test = series[9000:, :n_steps], series[9000:, -1]

In [15]:
model = tf.keras.models.Sequential([
    tf.keras.layers.SimpleRNN(1, input_shape=[None, 1])
])

/home/juanvieira/local/tf/env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [16]:
model.compile(optimizer="adam", loss="mse", metrics=['mae', 'mse'])

In [17]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3 (12.00 B)

 Trainable params: 3 (12.00 B)

 Non-trainable params: 0 (0.00 B)

In [18]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=30
)

Epoch 1/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 1.4823 - mae: 1.0788 - mse: 1.4823 - val_loss: 1.3414 - val_mae: 1.0361 - val_mse: 1.3414
Epoch 2/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 1.3064 - mae: 1.0176 - mse: 1.3064 - val_loss: 1.1995 - val_mae: 0.9837 - val_mse: 1.1995
Epoch 3/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 1.1614 - mae: 0.9633 - mse: 1.1614 - val_loss: 1.0976 - val_mae: 0.9441 - val_mse: 1.0976
Epoch 4/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 1.0834 - mae: 0.9323 - mse: 1.0834 - val_loss: 1.0156 - val_mae: 0.9108 - val_mse: 1.0156
Epoch 5/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.9911 - mae: 0.8955 - mse: 0.9911 - val_loss: 0.9460 - val_mae: 0.8815 - val_mse: 0.9460
Epoch 6/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - loss: 0.9204 - mae: 0.8641 - mse: 0.9204 - val_loss: 0.8840 - val_mae: 0.8543 - val_mse: 0.8840
Epoch 7/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.8412 - mae: 0.8267 - mse: 0.841